In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")
cal_data = pd.read_csv("m5-forecasting-accuracy/calendar.csv")
train_val_data = pd.read_csv("m5-forecasting-accuracy/sales_train_validation.csv")

print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

Training data shape: (1437, 1915)
Test data shape: (1437, 30)


In [3]:
print(cal_data.columns.tolist())

['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']


In [3]:
def melt_data_to_long_format(data, day_columns):
    """
    Transform wide format (products × days) to long format (product-day rows)
    """
    # Select relevant columns
    id_cols = ['id', 'dept_id']
    long_data = data[id_cols + day_columns].copy()
    
    # Melt day columns
    long_data = long_data.melt(
        id_vars=id_cols,
        value_vars=day_columns,
        var_name='day',
        value_name='sales'
    )
    
    return long_data

# Define day columns
train_days = [f'd_{i}' for i in range(1, 1914)]
test_days = [f'd_{i}' for i in range(1914, 1942)]

# Transform to long format
train_long = melt_data_to_long_format(train_data, train_days)
test_long = melt_data_to_long_format(test_data, test_days)

print(f"Training long format shape: {train_long.shape}")
print(f"Test long format shape: {test_long.shape}")

# === COMBINE DATA FOR PROPER LAG FEATURES ===
# Combine train and test data so test can access training history for lag features
combined_long = pd.concat([train_long, test_long], ignore_index=True)
 
# Sort by product and day to ensure proper time order
combined_long = combined_long.sort_values(['id', 'day'])
 
print(f"Combined long format shape: {combined_long.shape}")
print(f"\nSample combined data:")
print(combined_long.head())

Training long format shape: (2748981, 4)
Test long format shape: (40236, 4)
Combined long format shape: (2789217, 4)

Sample combined data:
                                  id  dept_id     day  sales
0        FOODS_1_001_CA_1_evaluation  FOODS_1     d_1      3
12933    FOODS_1_001_CA_1_evaluation  FOODS_1    d_10      0
142263   FOODS_1_001_CA_1_evaluation  FOODS_1   d_100      0
1435563  FOODS_1_001_CA_1_evaluation  FOODS_1  d_1000      1
1437000  FOODS_1_001_CA_1_evaluation  FOODS_1  d_1001      0


In [4]:
def create_features(long_data, calendar_data, price_data=None, is_train=True):
    """
    Create features for LightGBM based on MD file requirements:
    - Lagged sales
    - Rolling statistics  
    - Price features
    - Calendar/event features
    """
    # Extract day number
    long_data['day_num'] = long_data['day'].str.extract('d_(\d+)').astype(int)
    
    # Merge with calendar data (add more columns this time)
    long_data = long_data.merge(
        calendar_data[['d', 'date', 'weekday', 'wday', 'month', 'year', 
                      'event_name_1', 'event_type_1', 'snap_CA', 'wm_yr_wk']],
        left_on='day',
        right_on='d',
        how='left'
    )
    
    # === CYCLICAL TIME FEATURES (DO THIS FIRST - before categorical conversion) ===
    # Day of year (365-day cycle)
    long_data['day_sin'] = np.sin(2 * np.pi * long_data['day_num'] / 365)
    long_data['day_cos'] = np.cos(2 * np.pi * long_data['day_num'] / 365)
    
    # Day of week (7-day cycle) - use numeric wday, not categorical weekday
    long_data['weekday_sin'] = np.sin(2 * np.pi * long_data['wday'] / 7)
    long_data['weekday_cos'] = np.cos(2 * np.pi * long_data['wday'] / 7)
    
    # Month (12-month cycle) - use numeric month before categorical conversion
    long_data['month_sin'] = np.sin(2 * np.pi * long_data['month'] / 12)
    long_data['month_cos'] = np.cos(2 * np.pi * long_data['month'] / 12)
    
    # === NOW CONVERT TO CATEGORICAL (after math operations) ===
    long_data['dept_id'] = long_data['dept_id'].astype('category')
    long_data['weekday'] = long_data['weekday'].astype('category')
    long_data['month'] = long_data['month'].astype('category')
    
    # Handle NaN values in categorical columns
    long_data['event_name_1'] = long_data['event_name_1'].fillna('no_event').astype('category')
    long_data['event_type_1'] = long_data['event_type_1'].fillna('no_event').astype('category')
    
    # === LAG FEATURES ===
    # Sort by product and day to ensure proper lag calculation
    long_data = long_data.sort_values(['id', 'day_num'])
    
    # Create lag features (sales from previous periods)
    long_data['lag_7'] = long_data.groupby('id')['sales'].shift(7)
    long_data['lag_14'] = long_data.groupby('id')['sales'].shift(14)
    long_data['lag_28'] = long_data.groupby('id')['sales'].shift(28)
    long_data['lag_365'] = long_data.groupby('id')['sales'].shift(365)
    
    # === ROLLING STATISTICS ===
    # Rolling means (helped in baseline: 28-day MA was best)
    long_data['rolling_mean_7'] = long_data.groupby('id')['sales'].transform(
        lambda x: x.rolling(7, min_periods=1).mean()
    )
    long_data['rolling_mean_14'] = long_data.groupby('id')['sales'].transform(
        lambda x: x.rolling(14, min_periods=1).mean()
    )
    long_data['rolling_mean_28'] = long_data.groupby('id')['sales'].transform(
        lambda x: x.rolling(28, min_periods=1).mean()
    )
    
    # Rolling standard deviations (capture volatility)
    long_data['rolling_std_7'] = long_data.groupby('id')['sales'].transform(
        lambda x: x.rolling(7, min_periods=1).std()
    )
    long_data['rolling_std_28'] = long_data.groupby('id')['sales'].transform(
        lambda x: x.rolling(28, min_periods=1).std()
    )
    
    # === PRICE FEATURES ===
    if price_data is not None:
        # Merge with price data
        long_data = long_data.merge(
            price_data[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
            on=['store_id', 'item_id', 'wm_yr_wk'],
            how='left'
        )
        
        # Price change features
        long_data['price_lag_1'] = long_data.groupby('id')['sell_price'].shift(1)
        long_data['price_change'] = long_data['sell_price'] - long_data['price_lag_1']
        long_data['price_pct_change'] = long_data['sell_price'] / long_data['price_lag_1'] - 1
    
    # === EVENT FEATURES ===
    # Binary indicator for any event
    long_data['has_event'] = (~long_data['event_name_1'].isna()).astype(int)
    
    # SNAP indicator
    long_data['is_snap'] = long_data['snap_CA'].fillna(0).astype(int)
    
    return long_data

# Create features on COMBINED data (so test gets historical lags)
combined_features = create_features(combined_long, cal_data, is_train=True)
 
# Split back into train and test based on day_num
train_features = combined_features[combined_features['day_num'] <= 1913]
test_features = combined_features[combined_features['day_num'] > 1913]
 
print(f"Training features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")
 
# Check if test features now have proper lag values
print(f"\nTest lag_7 NaN count: {test_features['lag_7'].isna().sum()}")
print(f"Test lag_28 NaN count: {test_features['lag_28'].isna().sum()}")


Training features shape: (2748981, 32)
Test features shape: (40236, 32)

Test lag_7 NaN count: 0
Test lag_28 NaN count: 0


In [5]:
# === PREPARE DATA FOR LIGHTGBM ===

# Define feature columns (exclude non-feature columns)
feature_cols = [col for col in train_features.columns if col not in ['id', 'dept_id', 'day', 'sales', 'date', 'd', 'event_name_1', 'event_type_1']]

print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Prepare training data - remove rows with NaN in target or key features
train_clean = train_features.dropna(subset=['sales'] + feature_cols)

# Split into train and validation (time-based split)
# Use last 28 days of training for validation
val_day_num = train_clean['day_num'].max() - 28
train_split = train_clean[train_clean['day_num'] <= val_day_num]
val_split = train_clean[train_clean['day_num'] > val_day_num]

print(f"Training samples: {len(train_split)}")
print(f"Validation samples: {len(val_split)}")

# Prepare features and target
X_train = train_split[feature_cols]
y_train = train_split['sales']
X_val = val_split[feature_cols]
y_val = val_split['sales']

# Handle categorical features for LightGBM
categorical_features = ['dept_id', 'weekday', 'month']
categorical_indices = [i for i, col in enumerate(feature_cols) if col in categorical_features]

print(f"Categorical feature indices: {categorical_indices}")

Number of features: 24
Feature columns: ['day_num', 'weekday', 'wday', 'month', 'year', 'snap_CA', 'wm_yr_wk', 'day_sin', 'day_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'lag_7', 'lag_14', 'lag_28', 'lag_365', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'has_event', 'is_snap']
Training samples: 2184240
Validation samples: 40236
Categorical feature indices: [1, 3]


In [ ]:
# === CHECK FOR EXISTING MODEL AND CONTINUE TRAINING ===
import os

model_path = 'lightgbm_retail_model.txt'

if os.path.exists(model_path):
    print(f"Found existing model at {model_path}")
    print("Loading model to continue training...")
    model = lgb.Booster(model_file=model_path)
    print("Model loaded successfully! Continuing training...")
    start_iteration = model.num_trees()
    print(f"Starting from iteration {start_iteration}")
else:
    print("No existing model found. Training from scratch...")
    model = None
    start_iteration = 0

# Create LightGBM datasets
train_dataset = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_indices)
val_dataset = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_indices, reference=train_dataset)

# === YOUR HYPERPARAMETERS ===
# Add your custom parameters here
params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.015,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
    # Add your custom parameters here
}

# Define callbacks for newer LightGBM versions
callbacks = [
    lgb.early_stopping(stopping_rounds=100),
    lgb.log_evaluation(period=100)
]

# Train the model (continue from existing model or start fresh)
print("Training LightGBM model...")
if model is not None:
    # Continue training from existing model
    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[train_dataset, val_dataset],
        num_boost_round=1000,
        init_model=model,  # Continue from existing model
        callbacks=callbacks
    )
else:
    # Train from scratch
    model = lgb.train(
        params,
        train_dataset,
        valid_sets=[train_dataset, val_dataset],
        num_boost_round=1000,
        callbacks=callbacks
    )

print("Training completed!")

# Save the updated model
model.save_model(model_path)
print(f"Model saved to {model_path}")

Found existing model at lightgbm_retail_model.txt
Loading model to continue training...
Model loaded successfully! Continuing training...
Starting from iteration 2980
Training LightGBM model...


In [9]:
# === EVALUATE ON VALIDATION DATA INSTEAD ===

# Use validation data we already prepared
val_predictions = model.predict(X_val)

# Calculate WAPE on validation
total_absolute_error = np.sum(np.abs(y_val - val_predictions))
total_actual_sales = np.sum(y_val)
val_wape = total_absolute_error / total_actual_sales

print(f"Validation WAPE: {val_wape:.3f} ({val_wape:.1%})")

# Compare with baselines (recalculate for same validation period)
print("\n=== Model Comparison (Validation Period) ===")
print(f"LightGBM Validation WAPE: {val_wape:.3f} ({val_wape:.1%})")
print(f"LightGBM Validation MAE: 1.157 (from training)")

Validation WAPE: 0.542 (54.2%)

=== Model Comparison (Validation Period) ===
LightGBM Validation WAPE: 0.542 (54.2%)
LightGBM Validation MAE: 1.157 (from training)


In [10]:
# === EVALUATE ON TEST DATA (NOW WITH PROPER LAG FEATURES) ===

# Prepare test features (only drop rows where sales is NaN)
test_clean = test_features.dropna(subset=['sales'])

print(f"Test samples for evaluation: {len(test_clean)}")

# Get test features and target
X_test = test_clean[feature_cols]
y_test = test_clean['sales']

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Make predictions
test_predictions = model.predict(X_test)

# Calculate WAPE (your main metric from MD file)
total_absolute_error = np.sum(np.abs(y_test - test_predictions))
total_actual_sales = np.sum(y_test)
test_wape = total_absolute_error / total_actual_sales

print(f"Test WAPE: {test_wape:.3f} ({test_wape:.1%})")

# Compare with baselines
print("\n=== Final Model Comparison ===")
print(f"LightGBM Test WAPE: {test_wape:.3f} ({test_wape:.1%})")
print(f"LightGBM Validation WAPE: 0.550 (55.0%)")
print(f"Seasonal Naive WAPE: 0.944 (94.4%)")
print(f"28-day Moving Average WAPE: 0.685 (68.5%)")

if test_wape < 0.685:
    improvement = (0.685 - test_wape) / 0.685 * 100
    print(f"\n🎉 LightGBM beats the best baseline by {improvement:.1f}%!")
else:
    print(f"\nLightGBM test performance: {test_wape:.1%}")

Test samples for evaluation: 40236
X_test shape: (40236, 24)
y_test shape: (40236,)
Test WAPE: 0.533 (53.3%)

=== Final Model Comparison ===
LightGBM Test WAPE: 0.533 (53.3%)
LightGBM Validation WAPE: 0.550 (55.0%)
Seasonal Naive WAPE: 0.944 (94.4%)
28-day Moving Average WAPE: 0.685 (68.5%)

🎉 LightGBM beats the best baseline by 22.2%!


In [1]:
# === SAVE TRAINED MODEL ===
model.save_model('lightgbm_retail_model.txt')
print("Model saved successfully!")

NameError: name 'model' is not defined

In [ ]:
# === LOAD SAVED MODEL FOR PREDICTIONS ONLY ===
# Use this when you want to make predictions without any training

import lightgbm as lgb
loaded_model = lgb.Booster(model_file='lightgbm_retail_model.txt')
print("Model loaded successfully!")

# Use loaded model for predictions
# test_predictions = loaded_model.predict(X_test)